## 03: ETL Pipeline


### Introduction

This notebook implements the **Data Preparation (ETL)** phase of the CRISP-DM workflow for the Divvy bike-share project. It takes the raw monthly CSVs consolidated by `01_data_collection.ipynb` and turns them into a clean, dimensionally-modeled PostgreSQL warehouse — the same (empty) star schema built in `02_database_design.ipynb`.

**The pipeline, file by file:**

1. **Discovers** every raw monthly CSV in `csv_master/`, sorted chronologically.
2. **Loads** each file with explicit string dtypes so IDs and names never get silently coerced into numbers.
3. **Cleans** the data — trims whitespace, drops rows with a missing or duplicate `ride_id`, parses timestamps, computes trip duration, defensively fills missing categorical values, and *flags* (never silently drops) geospatial and duration outliers.
4. **Engineers the features** the warehouse needs: `start_hour`, `day_of_week`, `month_partition`, `start_date_key`/`end_date_key`, `is_round_trip`, `is_anomalous`.
5. **Builds dimension rows** (`dim_date`, `dim_station`) directly from that file's cleaned data.
6. **Upserts** those dimension rows into Postgres, then **resolves natural keys to surrogate keys** (station, ride-type, member-type) by reading them back from the warehouse.
7. **Bulk-loads** the resulting fact rows through a staging table + `COPY`, then merges into `fact_trip` with `ON CONFLICT (ride_id) DO NOTHING` — so re-running the notebook is always safe and idempotent.
8. **Repeats** steps 2–7 for every file inside a per-file try/except, so one bad month can never crash the whole run, and writes a data-quality report + log for every file processed.
9. Finishes with **post-load validation**: row counts per table, referential-integrity checks, and the date range actually loaded.
10. **Cleans up** the intermediate `downloads/` and `csv_master/` folders, but only after every file is confirmed loaded and validated, and records each ingested month in a manifest so the next collection run skips it.

**Why this matters (for recruiters/reviewers):** this notebook is a compact example of production-grade batch ETL — idempotent loads, defensive cleaning that quantifies every anomaly instead of silently discarding it, surrogate-key resolution against a live warehouse, bulk loading via `COPY` instead of row-by-row inserts, and automatic data-quality reporting with structured logging. These are the same patterns used in real data-engineering pipelines that run on a schedule against a production warehouse.

**Skills demonstrated:** ETL pipeline design, pandas-based data cleaning & feature engineering, dimensional (star-schema) modeling, SQL upserts, bulk data loading with PostgreSQL `COPY`, idempotent/incremental pipeline design, defensive error handling & data-quality reporting, structured logging.


### Import Libraries

Combines the standard library (`glob`, `logging`, `os`, `datetime`, `io.StringIO`, `pathlib.Path`) with three data/database-focused third-party packages: `numpy`/`pandas` for the cleaning and feature-engineering work, `psycopg2` (plus its `extras` module) for the PostgreSQL connection and bulk `execute_values`/`COPY` operations, and `tqdm` for a progress bar across the 32 monthly files. The local `config` module — shared with `02_database_design.ipynb` — keeps database credentials out of the notebook entirely.

Reusing `config.config_divvy()` instead of hardcoding a connection string means every notebook in the pipeline draws from the same credentials file, and nothing sensitive ever gets committed to source control alongside the analysis code.

**Skills demonstrated:** dependency management, secure credential handling (config separated from code), working with PostgreSQL from Python.


In [ ]:
import glob
import logging
import os
import shutil
from datetime import datetime
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
import psycopg2.extras as pg_extras
from tqdm import tqdm

import config

### Configuration

Centralizes every constant the rest of the notebook depends on: file-system paths (derived from the notebook's own location, not hardcoded), the expected raw column list and which of those columns should be treated as strings, a loose Chicagoland bounding box used only to *flag* — never silently drop — geospatial outliers, and thresholds for flagging suspiciously short (< 60s) or long (> 24h) trips. It also wires up file-based *and* console logging so every pipeline run leaves an audit trail in `etl_pipeline.log`.

Keeping "what counts as an anomaly" as named constants (`FLAG_SHORT_SECONDS`, `FLAG_LONG_HOURS`, `LAT_RANGE`/`LNG_RANGE`) instead of magic numbers buried in the cleaning logic makes the business rules easy to review, tune, and justify to a stakeholder without ever touching the cleaning code itself. Deriving `PROJECT_FOLDER` from the working directory also keeps the notebook portable across machines.

**Skills demonstrated:** configuration-driven design, structured logging, separating business rules from processing logic.


In [ ]:
# Project folder (comprehensive_divvy_bike_sharing_analytics), one level up from notebooks/
PROJECT_FOLDER = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Folder containing raw monthly CSVs produced by 01_data_collection.ipynb
BASE_FOLDER = os.path.join(PROJECT_FOLDER, "data")
os.makedirs(BASE_FOLDER, exist_ok=True)

CSV_FOLDER = os.path.join(BASE_FOLDER, "csv_master")

# Glob pattern matching Divvy's monthly export naming convention, e.g. 202401-divvy-tripdata.csv
CSV_PATTERN = "*-divvy-tripdata.csv"

# Raw columns every Divvy monthly export is expected to contain
RAW_COLUMNS = [
    "ride_id", "rideable_type", "started_at", "ended_at",
    "start_station_name", "start_station_id", "end_station_name", "end_station_id",
    "start_lat", "start_lng", "end_lat", "end_lng", "member_casual",
]
STRING_COLS = [
    "ride_id", "rideable_type", "start_station_name", "start_station_id",
    "end_station_name", "end_station_id", "member_casual",
]

# Loose Chicagoland bounding box -- used only to FLAG geospatial outliers, never to drop rows
LAT_RANGE = (41.55, 42.15)
LNG_RANGE = (-88.10, -87.40)

FLAG_SHORT_SECONDS = 60   # trips under 60s are likely false starts / immediate redocks
FLAG_LONG_HOURS = 24      # trips over 24h are likely lost, stolen, or improperly docked bikes

# Project folder (comprehensive_divvy_bike_sharing_analytics), one level up from notebooks/
PROJECT_FOLDER = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
REPORTS_FOLDER = os.path.join(PROJECT_FOLDER, "reports", "etl_reports")
os.makedirs(REPORTS_FOLDER, exist_ok=True)

REPORT_PATH = os.path.join(REPORTS_FOLDER, "etl_data_quality_report.csv")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(REPORTS_FOLDER, "etl_pipeline.log")),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("divvy_etl")

### Database Connection

A single small helper that opens a `psycopg2` connection to `divvy_db` using the same `config.config_divvy()` credentials as `02_database_design.ipynb`, with `autocommit` disabled so every file's load can be committed — or rolled back — as one atomic transaction. Any connection failure is logged and returns `None` rather than raising, letting the caller decide how to handle it.

Turning off autocommit is what makes the per-file `commit()`/`rollback()` pattern in `process_file()` (Step 5) possible: if a file's fact-table load fails partway through, every dimension upsert and partial fact insert for that file rolls back cleanly instead of leaving the warehouse in a half-loaded state.

**Skills demonstrated:** transactional database design, connection-level error handling.


In [ ]:
def connect_divvy():
    """Connect to the divvy_db database (mirrors 02_database_design.ipynb)."""
    conn = None
    try:
        params = config.config_divvy()
        conn = psycopg2.connect(**params)
        conn.autocommit = False
        return conn
    except (Exception, psycopg2.DatabaseError) as error:
        logger.error(f"Could not connect to divvy_db: {error}")
        return None

### Discover Source Files

Uses `pathlib.Path.glob()` with the `*-divvy-tripdata.csv` pattern to find every raw monthly file. Sorting alphabetically also sorts them chronologically, thanks to Divvy's `YYYYMM` filename prefix. If no files are found, it raises a clear `FileNotFoundError` that points back to `01_data_collection.ipynb`, instead of failing later with a confusing empty-dataframe error.

Fail-fast, descriptive errors at the very start of a pipeline save far more debugging time than a cryptic failure several steps downstream. Sorting by filename rather than modification time also guarantees a reproducible, deterministic processing order regardless of when files were downloaded.

**Skills demonstrated:** filesystem globbing, fail-fast validation, deterministic pipeline ordering.


In [ ]:
def discover_csv_files(csv_folder: str) -> list[Path]:
    """Find every raw monthly CSV, sorted chronologically by filename (YYYYMM prefix)."""
    paths = sorted(Path(csv_folder).glob(CSV_PATTERN))
    if not paths:
        raise FileNotFoundError(
            f"No files matching '{CSV_PATTERN}' found in {csv_folder}. "
            "Run 01_data_collection.ipynb first, or update CSV_FOLDER above."
        )
    logger.info(f"Discovered {len(paths)} source files in {csv_folder}")
    return paths

### Load & Clean

Turns one raw monthly CSV into an analysis-ready dataframe in two stages: a straightforward **raw load** followed by a much heavier **clean + feature-engineer** pass that is really the core of the whole notebook.


#### Raw load

Reads a single CSV with every ID/name/category column explicitly typed as pandas' nullable `string` dtype — never inferred as numeric — validates that all 13 expected Divvy columns are present, and stamps every row with its source filename (`__source_file`) for traceability back to the exact monthly export it came from.

Explicit dtypes prevent a classic silent-corruption bug: e.g. a `start_station_id` like `"620"` being auto-inferred as an integer and later failing to join against a station ID stored as text elsewhere. Validating columns before touching the data means a schema change in a future Divvy export fails loudly here, at load time, instead of producing a confusing `KeyError` several functions later.

**Skills demonstrated:** explicit schema enforcement, defensive column validation, data lineage/traceability.


In [ ]:
def load_raw_csv(path: Path) -> pd.DataFrame:
    """Read one Divvy monthly CSV with explicit dtypes."""
    dtype = {c: "string" for c in STRING_COLS}
    df = pd.read_csv(path, dtype=dtype)
    missing = set(RAW_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} is missing expected columns: {missing}")
    df = df[RAW_COLUMNS].copy()
    df["__source_file"] = path.name
    return df

#### Clean + feature-engineer

The heart of the ETL. In order, it: trims whitespace and normalizes empty strings to real nulls; drops rows with a missing `ride_id` (would violate the fact table's primary key); drops duplicate `ride_id`s within the file; parses `started_at`/`ended_at` and drops rows where either fails to parse; computes `duration_minutes` rounded to 2dp *before* filtering, so sub-hundredth-of-a-minute trips can't slip past the database's `duration_minutes > 0` check constraint after rounding; defensively fills missing `rideable_type`/`member_casual` with `"unknown"` so the fact table's `NOT NULL` foreign keys never break; coerces lat/lng to numeric; and finally engineers every derived column the warehouse needs (`start_hour`, `day_of_week`, `month_partition`, `start_date_key`, `end_date_key`, `is_round_trip`, `is_anomalous`). Every drop and every flag is counted into a `stats` dict that becomes one row of the end-of-run data-quality report.

Two design choices stand out. First, **anomalies are flagged, not dropped** — a trip with wildly out-of-bounds coordinates or an implausible duration is still loaded and analyzable, just tagged `is_anomalous = True`, so downstream analysis (`04_data_quality_assessment.ipynb`, `05_exploratory_analysis.ipynb`) can decide whether to exclude it rather than having that decision baked irreversibly into the ETL step. Second, the rounding-before-filtering order for `duration_minutes` is a subtle but important correctness fix: rounding *after* filtering would let a 0.00-minute trip (technically > 0 seconds, but rounding to 0.00) violate the database's positive-duration constraint at load time.

**Skills demonstrated:** data cleaning at scale with vectorized pandas operations, defensive NULL handling, constraint-aware data typing, feature engineering, root-cause debugging of edge cases (rounding order), quantified data-quality tracking.


In [ ]:
def clean_trips(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Clean and feature-engineer a raw trips dataframe.

    Returns (clean_df, stats) where stats is a dict of QA counters used to build
    the end-of-run data-quality report.
    """
    stats = {"rows_in": len(df)}

    # Trim whitespace on string fields; normalize empty strings to NA
    for col in STRING_COLS:
        df[col] = df[col].str.strip()
        df.loc[df[col] == "", col] = pd.NA

    # Drop rows with no ride_id (can't dedupe / violates the fact table's PK)
    before = len(df)
    df = df[df["ride_id"].notna()]
    stats["dropped_missing_ride_id"] = before - len(df)

    # Drop exact duplicate ride_ids within this file (keep first occurrence)
    before = len(df)
    df = df.drop_duplicates(subset="ride_id", keep="first")
    stats["dropped_duplicate_ride_id"] = before - len(df)

    # Parse timestamps -- handles both "YYYY-MM-DD HH:MM:SS" and
    # "YYYY-MM-DD HH:MM:SS.ffffff" formats seen across different export months
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df["ended_at"] = pd.to_datetime(df["ended_at"], errors="coerce")
    before = len(df)
    df = df[df["started_at"].notna() & df["ended_at"].notna()]
    stats["dropped_unparseable_timestamps"] = before - len(df)

    # Duration must be strictly positive to satisfy the DB CHECK constraints. Round
    # to 2dp FIRST (matching the fact_trip numeric(10,2) column) since sub-hundredth-
    # of-a-minute trips (e.g. 0.285 seconds) are non-positive *after* rounding even
    # though the raw duration is technically > 0 -- rounding after filtering would
    # let a few 0.00-minute rows slip through and violate the DB CHECK constraint.
    duration_minutes = ((df["ended_at"] - df["started_at"]).dt.total_seconds() / 60.0).round(2)
    before = len(df)
    keep = duration_minutes > 0
    stats["dropped_non_positive_duration"] = int((~keep).sum())
    df = df[keep].copy()
    df["duration_minutes"] = duration_minutes[keep]

    # Defensive fill so NOT NULL dimension foreign keys never break on ingest
    stats["filled_missing_rideable_type"] = int(df["rideable_type"].isna().sum())
    stats["filled_missing_member_casual"] = int(df["member_casual"].isna().sum())
    df["rideable_type"] = df["rideable_type"].fillna("unknown")
    df["member_casual"] = df["member_casual"].fillna("unknown")

    # Coerce lat/lng to numeric (guards against stray text values)
    for col in ["start_lat", "start_lng", "end_lat", "end_lng"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- Feature engineering required by divvy.fact_trip ---
    df["start_hour"] = df["started_at"].dt.hour.astype("int64")
    df["day_of_week"] = df["started_at"].dt.dayofweek + 1  # 1=Monday .. 7=Sunday
    df["month_partition"] = (df["started_at"].dt.year * 100 + df["started_at"].dt.month).astype("int64")
    df["start_date_key"] = df["started_at"].dt.strftime("%Y%m%d").astype("int64")
    df["end_date_key"] = df["ended_at"].dt.strftime("%Y%m%d").astype("int64")

    has_both_stations = df["start_station_id"].notna() & df["end_station_id"].notna()
    df["is_round_trip"] = pd.NA
    df.loc[has_both_stations, "is_round_trip"] = (
        df.loc[has_both_stations, "start_station_id"] == df.loc[has_both_stations, "end_station_id"]
    )

    out_of_bounds = (
        ~df["start_lat"].between(*LAT_RANGE) | ~df["start_lng"].between(*LNG_RANGE)
        | (df["end_lat"].notna() & ~df["end_lat"].between(*LAT_RANGE))
        | (df["end_lng"].notna() & ~df["end_lng"].between(*LNG_RANGE))
    )
    too_short = df["duration_minutes"] < (FLAG_SHORT_SECONDS / 60.0)
    too_long = df["duration_minutes"] > (FLAG_LONG_HOURS * 60)
    df["is_anomalous"] = (out_of_bounds | too_short | too_long).fillna(False)

    stats["flagged_anomalous"] = int(df["is_anomalous"].sum())
    stats["flagged_short_trips"] = int(too_short.sum())
    stats["flagged_long_trips"] = int(too_long.sum())
    stats["flagged_out_of_bounds_coords"] = int(out_of_bounds.fillna(False).sum())
    stats["rows_out"] = len(df)

    return df, stats

### Build Dimension Rows

Two builder functions generate this file's contribution to the warehouse's dimension tables. `build_dim_date` spans every calendar day between the earliest and latest `started_at`/`ended_at` in the batch and derives year/quarter/month/day/weekday/weekend flags for each. `build_dim_station` stacks every trip's start *and* end station into one table, then resolves one canonical row per `station_id` by taking the **statistical mode** of the station name (handles stations renamed mid-year) and the **median** lat/lng (smooths out GPS jitter across many trips at the same station).

Using mode/median instead of "first seen" or "last seen" makes the station dimension resilient to noisy or inconsistent source data — a handful of mistyped station names or slightly-off GPS pings won't corrupt the canonical record, because the majority/central value wins.

**Skills demonstrated:** dimensional modeling, statistical aggregation (mode/median) for deduplication, robust handling of noisy real-world data.


In [ ]:
def build_dim_date(clean_df: pd.DataFrame) -> pd.DataFrame:
    """Generate a full calendar covering every start/end date seen in this batch."""
    min_d = min(clean_df["started_at"].min(), clean_df["ended_at"].min()).normalize()
    max_d = max(clean_df["started_at"].max(), clean_df["ended_at"].max()).normalize()
    dates = pd.date_range(min_d, max_d, freq="D")
    dim_date = pd.DataFrame({"full_date": dates})
    dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype("int64")
    dim_date["year"] = dim_date["full_date"].dt.year
    dim_date["quarter"] = dim_date["full_date"].dt.quarter
    dim_date["month"] = dim_date["full_date"].dt.month
    dim_date["day"] = dim_date["full_date"].dt.day
    dim_date["day_of_week"] = dim_date["full_date"].dt.dayofweek + 1  # 1=Monday .. 7=Sunday
    dim_date["day_name"] = dim_date["full_date"].dt.day_name()
    dim_date["month_name"] = dim_date["full_date"].dt.month_name()
    dim_date["is_weekend"] = dim_date["day_of_week"].isin([6, 7])
    cols = ["date_key", "full_date", "year", "quarter", "month", "day",
            "day_of_week", "day_name", "month_name", "is_weekend"]
    return dim_date[cols]


def build_dim_station(clean_df: pd.DataFrame) -> pd.DataFrame:
    """Resolve one canonical (name, lat, lng) row per station_id from both trip endpoints."""
    starts = clean_df[["start_station_id", "start_station_name", "start_lat", "start_lng"]].rename(
        columns={"start_station_id": "station_id", "start_station_name": "station_name",
                 "start_lat": "lat", "start_lng": "lng"})
    ends = clean_df[["end_station_id", "end_station_name", "end_lat", "end_lng"]].rename(
        columns={"end_station_id": "station_id", "end_station_name": "station_name",
                 "end_lat": "lat", "end_lng": "lng"})
    all_st = pd.concat([starts, ends], ignore_index=True)
    all_st = all_st[all_st["station_id"].notna()]

    def _mode_or_na(s):
        m = s.dropna().mode()
        return m.iat[0] if not m.empty else pd.NA

    name_mode = all_st.groupby("station_id")["station_name"].agg(_mode_or_na)
    coords = all_st.groupby("station_id")[["lat", "lng"]].median()
    dim_station = name_mode.to_frame().join(coords).reset_index()
    dim_station.columns = ["station_id", "station_name", "latitude", "longitude"]
    return dim_station

### Load to PostgreSQL

Everything from here on writes to the live warehouse. Two groups of functions: **dimension upserts** that write/merge reference data, and **fact assembly & bulk load** that resolves natural keys to surrogate keys and inserts the actual trip records.


#### Dimension Upsert Functions

Four `upsert_*` helpers plus one `fetch_key_maps` helper. `upsert_dim_date`, `upsert_dim_ride_type`, and `upsert_dim_member_type` all use `ON CONFLICT ... DO NOTHING`, since those dimensions never need to change once a value exists. `upsert_dim_station` instead uses `ON CONFLICT ... DO UPDATE` with `COALESCE`, so if a station's name or coordinates are refined in a later month's data, the dimension is enriched without ever overwriting a good value with a NULL. All four use `psycopg2.extras.execute_values` to batch every row into one INSERT round-trip instead of one round-trip per row. `fetch_key_maps` then reads every dimension back out of the warehouse to build the natural-key → surrogate-key lookup tables the fact load needs next.

Batched `execute_values` inserts are dramatically faster than looping row-by-row for tables that can hold thousands of stations and hundreds of thousands of dates across a multi-year load. The `COALESCE`-based upsert for stations is a deliberate defensive choice: it lets the dimension self-heal as better data arrives without ever regressing to a worse value.

**Skills demonstrated:** SQL upsert patterns (`ON CONFLICT DO NOTHING` vs. `DO UPDATE`), batch/bulk database writes, surrogate-key management in a star schema.


In [ ]:
def upsert_dim_date(conn, dim_date: pd.DataFrame) -> None:
    sql = """
        INSERT INTO divvy.dim_date
            (date_key, full_date, year, quarter, month, day, day_of_week, day_name, month_name, is_weekend)
        VALUES %s
        ON CONFLICT (date_key) DO NOTHING;
    """
    rows = list(dim_date.itertuples(index=False, name=None))
    with conn.cursor() as cur:
        pg_extras.execute_values(cur, sql, rows)


def upsert_dim_station(conn, dim_station: pd.DataFrame) -> None:
    sql = """
        INSERT INTO divvy.dim_station (station_id, station_name, latitude, longitude)
        VALUES %s
        ON CONFLICT (station_id) DO UPDATE
            SET station_name = COALESCE(EXCLUDED.station_name, divvy.dim_station.station_name),
                latitude     = COALESCE(EXCLUDED.latitude, divvy.dim_station.latitude),
                longitude    = COALESCE(EXCLUDED.longitude, divvy.dim_station.longitude);
    """
    rows = [tuple(None if pd.isna(v) else v for v in r)
            for r in dim_station.itertuples(index=False, name=None)]
    with conn.cursor() as cur:
        pg_extras.execute_values(cur, sql, rows)


def upsert_dim_ride_type(conn, values: list[str]) -> None:
    sql = "INSERT INTO divvy.dim_ride_type (rideable_type) VALUES %s ON CONFLICT (rideable_type) DO NOTHING;"
    with conn.cursor() as cur:
        pg_extras.execute_values(cur, sql, [(v,) for v in values])


def upsert_dim_member_type(conn, values: list[str]) -> None:
    sql = "INSERT INTO divvy.dim_member_type (member_casual) VALUES %s ON CONFLICT (member_casual) DO NOTHING;"
    with conn.cursor() as cur:
        pg_extras.execute_values(cur, sql, [(v,) for v in values])


def fetch_key_maps(conn) -> dict:
    """Pull the current natural-key -> surrogate-key mappings back from the warehouse."""
    with conn.cursor() as cur:
        cur.execute("SELECT station_id, station_key FROM divvy.dim_station")
        station = pd.DataFrame(cur.fetchall(), columns=["station_id", "station_key"])
        cur.execute("SELECT rideable_type, ride_type_key FROM divvy.dim_ride_type")
        ride_type = pd.DataFrame(cur.fetchall(), columns=["rideable_type", "ride_type_key"])
        cur.execute("SELECT member_casual, member_type_key FROM divvy.dim_member_type")
        member_type = pd.DataFrame(cur.fetchall(), columns=["member_casual", "member_type_key"])
    return {"station": station, "ride_type": ride_type, "member_type": member_type}

#### Fact Table Assembly & Bulk Load

`build_fact_dataframe` merges the cleaned trip data against every key map from 4.1 to swap natural keys (`start_station_id`, `rideable_type`, `member_casual`, ...) for the surrogate keys `fact_trip` actually expects, then fails loudly with a `ValueError` if any row can't resolve a `NOT NULL` foreign key (`ride_type_key`/`member_type_key`) rather than silently loading a broken row. It also explicitly recasts every integer-typed fact column to pandas' nullable `Int64`, because a left-merge introduces float `NaN` for unmatched keys, which silently upcasts a clean integer column to `float64` (turning `820` into `"820.0"`) — and Postgres' `COPY` rejects that malformed integer text outright.

`load_fact_trip` then does the actual bulk insert: it creates a temporary staging table shaped like `fact_trip` (minus the auto-generated surrogate key), streams the whole dataframe into it in one shot via `COPY ... FROM STDIN` using an in-memory `StringIO` buffer instead of a temp file on disk, and finally merges staging into the real fact table with `ON CONFLICT (ride_id) DO NOTHING` — so re-running the notebook on a file that's already loaded inserts zero duplicate rows.

`COPY` is one to two orders of magnitude faster than row-by-row `INSERT` for loading hundreds of thousands of rows per file, which matters here since several monthly files exceed 800,000 trips. The temp staging table + `ON CONFLICT` pattern gives idempotency "for free" without needing to check for existing `ride_id`s in Python first, and catching the float-upcast-on-merge bug before it ever reaches `COPY` is exactly the kind of subtle production issue that separates a notebook that "mostly works" from a pipeline that's actually safe to re-run.

**Skills demonstrated:** bulk data loading with PostgreSQL `COPY`, staging-table load pattern, idempotent upsert design, dataframe dtype management for database compatibility, fail-fast referential-integrity validation.


In [ ]:
def build_fact_dataframe(clean_df: pd.DataFrame, key_maps: dict) -> pd.DataFrame:
    """Resolve natural keys to surrogate keys and assemble the fact_trip-shaped dataframe."""
    fact = clean_df.merge(
        key_maps["station"], left_on="start_station_id", right_on="station_id", how="left"
    ).rename(columns={"station_key": "start_station_key"}).drop(columns=["station_id"])

    fact = fact.merge(
        key_maps["station"], left_on="end_station_id", right_on="station_id", how="left"
    ).rename(columns={"station_key": "end_station_key"}).drop(columns=["station_id"])

    fact = fact.merge(key_maps["ride_type"], on="rideable_type", how="left")
    fact = fact.merge(key_maps["member_type"], on="member_casual", how="left")

    # ride_type_key / member_type_key are NOT NULL on fact_trip -- if a lookup fails
    # (e.g. dimension upsert happened out of order), fail loudly rather than silently
    # loading bad rows.
    n_missing = fact["ride_type_key"].isna().sum() + fact["member_type_key"].isna().sum()
    if n_missing:
        raise ValueError(f"{n_missing} rows failed to resolve ride_type_key/member_type_key")

    fact_cols = [
        "ride_id", "start_date_key", "end_date_key", "start_station_key", "end_station_key",
        "ride_type_key", "member_type_key", "started_at", "ended_at", "duration_minutes",
        "start_hour", "day_of_week", "month_partition",
        "start_lat", "start_lng", "end_lat", "end_lng",
        "is_round_trip", "is_anomalous",
    ]
    fact_final = fact[fact_cols].copy()
    fact_final["is_round_trip"] = fact_final["is_round_trip"].astype("object").where(
        fact_final["is_round_trip"].notna(), None
    )

    # Merges introduce float NaN for unmatched keys, which silently upcasts integer
    # columns to float64 (e.g. 820 -> "820.0"). Postgres' integer COPY rejects "820.0",
    # so cast every integer-typed fact column to pandas' nullable Int64 to preserve
    # NULLs without adding a decimal point.
    int_cols = [
        "start_date_key", "end_date_key", "start_station_key", "end_station_key",
        "ride_type_key", "member_type_key", "start_hour", "day_of_week", "month_partition",
    ]
    for col in int_cols:
        fact_final[col] = fact_final[col].astype("Int64")

    return fact_final


def load_fact_trip(conn, fact_df: pd.DataFrame) -> int:
    """Bulk-load fact rows via COPY into a temp staging table, then upsert into fact_trip.

    Returns the number of NEW rows actually inserted (existing ride_ids are skipped).
    """
    fact_cols = list(fact_df.columns)
    with conn.cursor() as cur:
        cur.execute(f"""
            CREATE TEMP TABLE staging_fact_trip (LIKE divvy.fact_trip INCLUDING DEFAULTS)
            ON COMMIT DROP;
            ALTER TABLE staging_fact_trip DROP COLUMN trip_key;
        """)

        buf = StringIO()
        fact_df.to_csv(buf, index=False, header=False, na_rep="\\N")
        buf.seek(0)
        cur.copy_expert(
            f"COPY staging_fact_trip ({', '.join(fact_cols)}) FROM STDIN WITH (FORMAT csv, NULL '\\N')",
            buf,
        )

        cur.execute(f"""
            INSERT INTO divvy.fact_trip ({', '.join(fact_cols)})
            SELECT {', '.join(fact_cols)} FROM staging_fact_trip
            ON CONFLICT (ride_id) DO NOTHING;
        """)
        return cur.rowcount

### Orchestration

Ties every function above together into a repeatable, resilient batch process: run one file through the full pipeline, then run every file through that same logic.


#### Per-File Processor

`process_file` runs the full clean → dimension-upsert → fact-load sequence for one CSV inside a try/except, committing the transaction only on full success and rolling it back on any error — so a single malformed month can never leave the warehouse in a partially-loaded state, and never stops the rest of the run. Every outcome (success, skipped-empty, or failed-with-error-message) is captured in a result dict that feeds the final data-quality report.

Wrapping each file's entire load in one transaction is what makes the whole pipeline safe to re-run after a partial failure — either a file's data is fully in the warehouse, or none of it is, with no silent middle ground.

**Skills demonstrated:** transactional integrity, per-unit error isolation (one bad file can't fail the batch), structured result/error reporting.


In [ ]:
def process_file(path: Path, conn) -> dict:
    """Run the full clean -> dimension-upsert -> fact-load flow for a single CSV file."""
    result = {"file": path.name, "status": "success", "error": None}
    try:
        raw = load_raw_csv(path)
        clean, clean_stats = clean_trips(raw)
        result.update(clean_stats)

        if clean.empty:
            result["status"] = "skipped_empty"
            result["fact_rows_inserted"] = 0
            return result

        dim_date = build_dim_date(clean)
        dim_station = build_dim_station(clean)
        ride_types = clean["rideable_type"].dropna().unique().tolist()
        member_types = clean["member_casual"].dropna().unique().tolist()

        upsert_dim_date(conn, dim_date)
        upsert_dim_station(conn, dim_station)
        upsert_dim_ride_type(conn, ride_types)
        upsert_dim_member_type(conn, member_types)

        key_maps = fetch_key_maps(conn)
        fact_df = build_fact_dataframe(clean, key_maps)
        inserted = load_fact_trip(conn, fact_df)

        conn.commit()
        result["fact_rows_inserted"] = inserted
        logger.info(f"{path.name}: cleaned {clean_stats['rows_out']:,} rows, inserted {inserted:,} new fact rows")

    except Exception as exc:
        conn.rollback()
        result["status"] = "failed"
        result["error"] = str(exc)
        logger.error(f"{path.name}: FAILED -- {exc}")

    return result

#### Pipeline Runner

`run_pipeline` opens one database connection, then walks every discovered file through `process_file` with a `tqdm` progress bar, guaranteeing the connection is always closed via `finally` even if the loop is interrupted partway through. It collects every per-file result into a summary dataframe and writes it to `etl_data_quality_report.csv` for later review.

Reusing a single open connection across all 32 files — rather than reconnecting per file — avoids unnecessary connection overhead, while still isolating failures at the per-file transaction level set up in 5.1. Persisting the summary report to disk means the data-quality outcome of a multi-hour run doesn't just live in a notebook's transient output — it survives to be reviewed, diffed against previous runs, or handed to a stakeholder.

**Skills demonstrated:** resource lifecycle management (`try`/`finally`), batch orchestration with progress tracking, automated reporting.


In [ ]:
def run_pipeline(csv_folder: str = CSV_FOLDER) -> pd.DataFrame:
    """Process every discovered CSV file and return a per-file summary report."""
    files = discover_csv_files(csv_folder)
    conn = connect_divvy()
    if conn is None:
        raise ConnectionError("Could not connect to divvy_db.")

    summary_rows = []
    try:
        for path in tqdm(files, desc="Cleaning & loading"):
            summary_rows.append(process_file(path, conn))
    finally:
        conn.close()

    report = pd.DataFrame(summary_rows)
    report.to_csv(REPORT_PATH, index=False)
    logger.info(f"Data-quality report written to {REPORT_PATH}")
    return report

### Run the Pipeline

Executes the full pipeline against all 32 discovered monthly files and prints a top-level summary. The run captured below processed all 32 files successfully with zero failures, cleaned roughly 15.67M of ~15.67M raw rows (under 0.01% dropped — almost entirely non-positive-duration trips), and shows `fact_rows_inserted = 0` for every file. That's expected rather than a bug: this run executed against a `divvy_db` warehouse that already held this data from an earlier load, so the `ON CONFLICT (ride_id) DO NOTHING` upsert correctly recognized every row as already present and skipped re-inserting it.

That "0 new rows on a repeat run" result is actually the pipeline's idempotency guarantee working exactly as designed — proof the notebook can be re-run safely (e.g. on a schedule, or after adding new months) without ever double-counting a trip.

**Skills demonstrated:** end-to-end pipeline execution, interpreting idempotent-load results, run-level summary reporting.


In [ ]:
report = run_pipeline()

n_failed = (report["status"] == "failed").sum()
print(f"Files processed : {len(report)}")
print(f"Succeeded        : {(report['status'] == 'success').sum()}")
print(f"Failed           : {n_failed}")
print(f"Total rows in    : {report.get('rows_in', pd.Series(dtype=int)).sum():,}")
print(f"Total rows out   : {report.get('rows_out', pd.Series(dtype=int)).sum():,}")
print(f"Total fact rows inserted : {report.get('fact_rows_inserted', pd.Series(dtype=int)).sum():,}")

if n_failed:
    print("\nFailed files:")
    print(report.loc[report['status'] == 'failed', ['file', 'error']])

report

### Post-Load Validation

Quick sanity checks that re-query the warehouse directly — independent of anything computed in Python during the load — to confirm the data actually landed correctly: row counts per table, an explicit `LEFT JOIN` check for orphaned fact rows that reference a station or date key with no matching dimension row, referential integrity, and the overall date range now covered by `fact_trip`. This is also how idempotency gets verified end-to-end: re-running the pipeline on the same files should insert 0 new rows the second time, which is exactly what Step 6 showed.

The result below confirms **zero** orphaned station/date references, a `dim_station` table of 3,943 stations, a `dim_date` table of 974 calendar days, and a `fact_trip` table of roughly 15.67M trips spanning January 2024 through August 2026.

Trusting that a load "probably worked" just because the pipeline didn't raise an exception isn't the same as verifying it referentially — orphaned foreign keys are a classic silent failure mode in star-schema warehouses (e.g. from a dimension upsert that ran after, instead of before, its dependent fact rows). Re-querying the warehouse from scratch, rather than reusing anything already held in memory, is what makes this a genuinely independent check.

**Skills demonstrated:** SQL-based data validation, referential-integrity testing, warehouse-level quality assurance independent of the ETL code path.


In [ ]:
def validate_load():
    conn = connect_divvy()
    checks = {}
    try:
        with conn.cursor() as cur:
            for table in ["dim_date", "dim_station", "dim_ride_type", "dim_member_type", "fact_trip"]:
                cur.execute(f"SELECT COUNT(*) FROM divvy.{table};")
                checks[f"{table}_row_count"] = cur.fetchone()[0]

            cur.execute("""
                SELECT COUNT(*) FROM divvy.fact_trip f
                LEFT JOIN divvy.dim_station s ON f.start_station_key = s.station_key
                WHERE f.start_station_key IS NOT NULL AND s.station_key IS NULL;
            """)
            checks["orphan_start_station_refs"] = cur.fetchone()[0]

            cur.execute("""
                SELECT COUNT(*) FROM divvy.fact_trip f
                LEFT JOIN divvy.dim_date d ON f.start_date_key = d.date_key
                WHERE d.date_key IS NULL;
            """)
            checks["orphan_start_date_refs"] = cur.fetchone()[0]

            cur.execute("SELECT MIN(started_at), MAX(started_at) FROM divvy.fact_trip;")
            checks["date_range_loaded"] = cur.fetchone()
    finally:
        conn.close()
    return checks

validation = validate_load()
validation

### Post-Load Cleanup

Once the warehouse holds the data, the raw ZIPs in `downloads/` and the consolidated CSVs in `csv_master/` are just rebuildable intermediates (tens of GB for 32+ months of trips). This step deletes them, but only if it can prove the load succeeded.

**Deletion gate (all must pass, otherwise nothing is deleted):**

1. No file in the pipeline report has a `failed` status.
2. Post-load validation found zero orphaned station/date references.
3. Every successfully processed month is actually present in `fact_trip` (checked against `month_partition` in the database itself, not in-memory state).

**Safety rails:** `CLEANUP_ENABLED` is a master switch; `DRY_RUN = True` reports what would be removed without deleting; only real directories directly inside `data/` can be deleted (symlinks and anything outside `data/` are refused); and all outcomes go through the same logger as the rest of the pipeline.

**Manifest:** before deleting, each ingested month is appended to `data/ingested_files.txt`. `01_data_collection.ipynb` reads that file and skips those months, so deleting the ZIPs does not cause the whole history to be re-downloaded and re-processed on the next run. If you ever wipe the database, delete the manifest too.

**Skills demonstrated:** verify-then-delete pipeline hygiene, defensive destructive operations, disk-space management, cross-notebook state via a lightweight ledger.

In [ ]:
# Cleanup configuration
CLEANUP_ENABLED = True      # master switch: False = never delete anything
DRY_RUN = False             # True = verify + report what WOULD be deleted, delete nothing

# Intermediate folders produced by 01_data_collection.ipynb that are safe to rebuild.
# (Add os.path.join(BASE_FOLDER, "extracted") here to reclaim that space as well.)
CLEANUP_FOLDERS = [
    os.path.join(BASE_FOLDER, "downloads"),   # raw monthly ZIPs
    CSV_FOLDER,                               # csv_master: consolidated monthly CSVs
]

# Ledger of months already loaded into the warehouse; 01_data_collection.ipynb reads it
# so deleted months are not downloaded / processed again on the next monthly run.
MANIFEST_PATH = os.path.join(BASE_FOLDER, "ingested_files.txt")
INGESTED_STATUSES = {"success", "skipped_empty"}   # skipped_empty = file had no valid rows


def verify_ingestion(report: pd.DataFrame, validation: dict) -> list[str]:
    """Return a list of problems; an empty list means it is safe to delete the sources."""
    problems = []

    if report.empty:
        return ["pipeline report is empty -- nothing was processed"]

    failed = report.loc[report["status"] == "failed", "file"].tolist()
    if failed:
        problems.append(f"{len(failed)} file(s) failed to load: {failed}")

    for key in ("orphan_start_station_refs", "orphan_start_date_refs"):
        if validation.get(key):
            problems.append(f"post-load validation: {key} = {validation[key]:,}")

    # Every successfully processed month must actually exist in fact_trip
    loaded_files = report.loc[report["status"] == "success", "file"]
    conn = connect_divvy()
    if conn is None:
        return problems + ["could not connect to divvy_db to confirm loaded months"]
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT month_partition FROM divvy.fact_trip GROUP BY month_partition;")
            months_in_db = {row[0] for row in cur.fetchall()}
    finally:
        conn.close()

    missing = [f for f in loaded_files if int(Path(f).name[:6]) not in months_in_db]
    if missing:
        problems.append(f"months not found in fact_trip: {missing}")

    return problems


def _is_safe_to_delete(folder: str) -> bool:
    """Only allow deleting a real (non-symlink) directory that sits directly inside data/."""
    if os.path.islink(folder):
        return False
    real_base = os.path.realpath(BASE_FOLDER)
    real_folder = os.path.realpath(folder)
    return os.path.dirname(real_folder) == real_base and real_folder != real_base


def _folder_size_mb(folder: str) -> float:
    return sum(p.stat().st_size for p in Path(folder).rglob("*") if p.is_file()) / 1024**2


def update_manifest(report: pd.DataFrame) -> int:
    """Record every ingested month (by file stem) so 01_data_collection.ipynb can skip it."""
    ingested = set()
    if os.path.exists(MANIFEST_PATH):
        ingested = set(Path(MANIFEST_PATH).read_text().split())
    before = len(ingested)
    ingested |= {Path(f).stem for f in report.loc[report["status"].isin(INGESTED_STATUSES), "file"]}
    Path(MANIFEST_PATH).write_text("\n".join(sorted(ingested)) + "\n")
    return len(ingested) - before


def cleanup_ingested_sources(report: pd.DataFrame, validation: dict) -> None:
    if not CLEANUP_ENABLED:
        logger.info("Cleanup disabled (CLEANUP_ENABLED = False); source folders kept.")
        return

    problems = verify_ingestion(report, validation)
    if problems:
        logger.warning("Cleanup SKIPPED -- source files kept so the load can be retried:")
        for p in problems:
            logger.warning(f"  - {p}")
        return

    tag = "[DRY RUN] " if DRY_RUN else ""
    if not DRY_RUN:
        added = update_manifest(report)          # record BEFORE deleting anything
        logger.info(f"Manifest updated: {added} new month(s) -> {MANIFEST_PATH}")

    freed_mb = 0.0
    for folder in CLEANUP_FOLDERS:
        if not os.path.isdir(folder):
            logger.info(f"{tag}Not found, nothing to delete: {folder}")
            continue
        if not _is_safe_to_delete(folder):
            logger.error(f"{tag}Refusing to delete (outside data/ or a symlink): {folder}")
            continue
        size_mb = _folder_size_mb(folder)
        if DRY_RUN:
            logger.info(f"{tag}Would delete {folder} ({size_mb:,.0f} MB)")
        else:
            shutil.rmtree(folder)
            logger.info(f"Deleted {folder} ({size_mb:,.0f} MB)")
        freed_mb += size_mb

    logger.info(f"{tag}Total space {'that would be ' if DRY_RUN else ''}freed: {freed_mb:,.0f} MB")


cleanup_ingested_sources(report, validation)

### Key Takeaways

This notebook turns 32 raw monthly CSVs into a fully-loaded, referentially-sound PostgreSQL star schema, and it does so the way a production batch-ETL job would be written:

- **Idempotent by design** — every dimension upsert and fact insert uses `ON CONFLICT`, so re-running the whole notebook (confirmed live in Step 6/7) inserts zero duplicate rows.
- **Defensive, quantified cleaning** — every dropped or flagged row is counted into a per-file stats dict and rolled up into a written data-quality report, instead of anomalies silently disappearing.
- **Anomalies flagged, never hidden** — out-of-bounds coordinates and implausible durations are tagged `is_anomalous` and kept, leaving the decision to exclude them to downstream analysis.
- **Transactional integrity per file** — one bad month rolls back cleanly and never corrupts the warehouse or halts the batch.
- **Built for scale** — bulk `COPY` loads and batched `execute_values` upserts, not row-by-row inserts, keep the pipeline fast across files with 800,000+ trips.
- **Independently verified** — post-load validation re-queries the warehouse directly rather than trusting in-memory pipeline state, catching referential-integrity issues a "no exception was raised" check would miss.
- **Verify-then-delete cleanup** — intermediate ZIPs and CSVs are removed only after every file is confirmed loaded and validated, with a dry-run mode, path safety checks, and a manifest that keeps future collection runs incremental.

**Skills on display:** end-to-end ETL pipeline design, pandas-based data cleaning & feature engineering, dimensional (star-schema) data modeling, SQL upserts and bulk loading (`COPY`), idempotent/transactional pipeline design, defensive error handling, automated data-quality reporting, and warehouse-level data validation — the same fundamentals used in real-world production data-engineering pipelines.
